# 02 - Agentic RAG Pipeline
## Planner -> Executor -> Synthesizer

This notebook demonstrates the 3-agent pipeline that powers InsightForge:
1. **Planner**: Uses LLM to create an analysis plan from user query + data profile
2. **Executor**: Runs pandas operations and selects chart types
3. **Synthesizer**: Crafts professional analyst-style narrative

### Architecture
```
User Query + CSV Profile
        |
   [Planner Agent] -- LLM --> Analysis Plan (JSON)
        |
   [Executor Agent] -- Pandas --> Results + Chart Config
        |
   [Synthesizer Agent] -- LLM --> Professional Narrative
        |
   Response (Story + Visualization)
```

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import json

from core.llm_client import LLMRouter
from agents.planner import PlannerAgent
from agents.executor import ExecutorAgent
from agents.synthesizer import SynthesizerAgent
from ingestion.csv_profiler import CSVProfiler
from tools.viz_tool import VizTool

print('All agents loaded successfully!')

## Step 1: Prepare Data
Load and profile a sample dataset, then clean it.

In [ ]:
# Create sample data
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'product_category': np.random.choice(['Electronics', 'Clothing', 'Books', 'Home', 'Sports'], n),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n),
    'revenue': np.random.uniform(10, 500, n).round(2),
    'quantity': np.random.randint(1, 20, n),
    'rating': np.random.uniform(1, 5, n).round(1),
})

# Save and profile
csv_path = '../data/sample_sales_clean.csv'
df.to_csv(csv_path, index=False)

profiler = CSVProfiler()
profile = profiler.profile(csv_path)

print(f'Dataset: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
df.head()

## Step 2: Initialize the LLM Router

The `LLMRouter` tries Groq (Llama 3.3 70B) first for speed,
then falls back to Google Gemini if Groq fails.

In [ ]:
# Initialize LLM Router (reads API keys from .env)
llm = LLMRouter()

# Quick test
response = llm.chat([{'role': 'user', 'content': 'Say hello in one word'}])
print(f'LLM Response: {response}')
print(f'Provider used: {llm.last_provider}')

## Step 3: Planner Agent

The Planner receives a user query + dataset profile and outputs a structured
JSON analysis plan: which columns to analyze, what aggregation, which chart type.

In [ ]:
planner = PlannerAgent(llm)

# Test with different queries
queries = [
    'What is the average revenue by product category?',
    'Show me the distribution of ratings',
    'Which region has the highest total revenue?',
    'Is there a correlation between quantity and revenue?',
]

for q in queries:
    plan = planner.plan(q, profile)
    print(f'\nQuery: "{q}"')
    print(f'Plan: {json.dumps(plan, indent=2)}')
    print('-' * 50)

## Step 4: Executor Agent

The Executor takes the plan and runs actual pandas operations.
It returns raw analysis results AND a Chart.js-compatible config.

In [ ]:
executor = ExecutorAgent()

# Execute the first query's plan
query = 'What is the average revenue by product category?'
plan = planner.plan(query, profile)
result = executor.execute(plan, df)

print('Analysis Result:')
print(json.dumps(result['analysis_result'], indent=2, default=str))

print(f'\nChart Type: {result["chart_config"]["type"]}')
print(f'Labels: {result["chart_config"]["data"]["labels"]}')

## Step 5: Synthesizer Agent

The Synthesizer takes the analysis results and crafts a professional,
McKinsey-style analyst narrative with key findings and recommendations.

In [ ]:
synthesizer = SynthesizerAgent(llm)

narrative = synthesizer.synthesize(
    user_query=query,
    plan=plan,
    analysis_result=result['analysis_result'],
    summary_data=result['summary_data'],
    profile=profile,
)

print('=== ANALYST NARRATIVE ===')
print(narrative)

## Step 6: Full Pipeline - End to End

Run the complete pipeline for a new query.

In [ ]:
def run_pipeline(query, df, profile, llm):
    """Run the complete Planner -> Executor -> Synthesizer pipeline."""
    planner = PlannerAgent(llm)
    executor = ExecutorAgent()
    synthesizer = SynthesizerAgent(llm)
    
    # Step 1: Plan
    plan = planner.plan(query, profile)
    print(f'Plan: {plan["analysis_type"]} | Viz: {plan["visualization"]}')
    
    # Step 2: Execute
    result = executor.execute(plan, df)
    
    # Step 3: Synthesize
    narrative = synthesizer.synthesize(query, plan, result['analysis_result'],
                                       result['summary_data'], profile)
    
    return {
        'plan': plan,
        'result': result,
        'narrative': narrative,
        'chart_config': result['chart_config'],
    }

# Run for a new query
output = run_pipeline('Compare revenue across regions', df, profile, llm)
print('\n' + output['narrative'])

## Step 7: Visualization Overview Generation

The `VizTool` generates BI-style overview charts including KPI cards,
horizontal bars, combo charts, radar charts, and more.

In [ ]:
viz_tool = VizTool()
overview = viz_tool.generate_overview_charts(df, profile)

print(f'Generated {len(overview)} overview charts:\n')
for chart in overview:
    chart_type = chart['config'].get('type', 'custom')
    print(f'  - {chart["title"]} ({chart_type})')

---
## Summary

This notebook demonstrated the complete agentic RAG pipeline:

| Agent | Role | Input | Output |
|-------|------|-------|--------|
| **Planner** | Decides analysis strategy | Query + Profile | JSON plan |
| **Executor** | Runs pandas operations | Plan + DataFrame | Results + Chart |
| **Synthesizer** | Crafts narrative | Results + Context | Analyst story |

The pipeline runs automatically when users chat in the web interface.
All visualizations are Chart.js-compatible and can be added to custom dashboards.